# Coupon Experiment Decision Framework — Walkthrough

This notebook walks through the framework stage by stage using simulated data.

Note: the dataset contains hidden latent fields for later validation. Stages 1-4 only use observable fields; latent truth and long-term outcomes are revealed in Stage 5.

> **Data note:** The simulated dataset includes latent fields for validation
> purposes. The walkthrough treats those fields as unavailable until the
> calibration stage. Stages 1 to 4 use only observable fields.

In [1]:
import pandas as pd

df = pd.read_csv("../data/simulated_coupon_experiment.csv")

# Stages 1-4 only use observable fields. The CSV also contains latent ground
# truth and long-term outcome columns, which are intentionally not loaded or
# shown until Stage 5.
observable_cols = [
    "user_id",
    "group",
    "coupon_received",
    "signup_day",
    "exposure_day",
    "made_first_purchase_30d",
    "days_to_first_purchase",
    "repeat_visits_14d",
    "browse_sessions_14d",
    "product_views_14d",
    "cart_adds_14d",
    "discount_page_views_14d",
    "coupon_used",
]

df[observable_cols].head()

,user_id,group,coupon_received,signup_day,exposure_day,made_first_purchase_30d,days_to_first_purchase,repeat_visits_14d,browse_sessions_14d,product_views_14d,cart_adds_14d,discount_page_views_14d,coupon_used
0,1,treatment,1,82,82,0,NaN,0,1,1,2,3,0
1,2,control,0,67,74,0,NaN,0,0,1,1,0,0
2,3,treatment,1,19,23,1,17.0,2,4,12,0,4,1
3,4,treatment,1,90,97,0,NaN,4,4,11,4,1,0
4,5,control,0,90,93,1,7.0,5,7,14,2,0,0


## Stage 1: Measurement Design

Stage 1 is about designing the experiment so it can answer a stronger question than "did the 30-day first-purchase rate go up?" — it should also capture what *kind* of value any lift represents. That means committing up front to the primary metric, guardrail metrics, holdout logic, observation windows, and the post-period tracking needed to later assess incremental demand, customer quality, and borrowed-demand risk.

### Trap 1: The Headline Lift

In [2]:
rates = (
    df.groupby("group")["made_first_purchase_30d"]
    .mean()
    .rename("first_purchase_rate_30d")
)

control_rate = rates.loc["control"]
treatment_rate = rates.loc["treatment"]
lift_pp = (treatment_rate - control_rate) * 100

print(f"Control   30d first-purchase rate: {control_rate:.2%}")
print(f"Treatment 30d first-purchase rate: {treatment_rate:.2%}")
print(f"Headline lift:                     {lift_pp:+.2f}pp")

rates.to_frame()

Control   30d first-purchase rate: 17.71%
Treatment 30d first-purchase rate: 23.55%
Headline lift:                     +5.84pp


,first_purchase_rate_30d
group,
control,0.177069
treatment,0.235507


The coupon appears to improve the 30-day first-purchase rate. But this only answers whether the rate went up. It does not tell us what kind of value that lift represents.

### Trap 2: Same Lift, Different Reality

| Scenario | Headline lift | Dominant hidden reality | Decision implication |
|----------|---------------|-------------------------|----------------------|
| A | +6pp | More persuadable high-value users | Consider scale |
| B | +6pp | More discount-driven and pulled-forward users | Adjust or stop |

Two experiments can show the same headline lift while representing very different underlying realities. The same +6pp could justify scaling in one case and stopping in another. This is the gap the rest of the framework aims to close. We will return to validate this with long-term outcomes in Stage 5.

## Stage 2: Early Signal Layer

While long-term outcomes are still immature, we turn raw early behaviour into
a small set of transparent, rule-based **derived signals**. Each signal is a
simple combination of *observable* fields only, expressed on a common 0–1
percentile scale so that fields with different units stay comparable:

- **engagement_intensity** — breadth of early activity (repeat visits, browse
  sessions, product views).
- **purchase_intent** — depth of intent (cart adds), kept separate so product
  views are not weighted twice.
- **discount_dependency** — reliance on the discount itself (discount-page
  views combined with whether a coupon was redeemed).
- **early_timing** — how early the first purchase landed (purchasers only; NaN
  otherwise). On its own this is *not* a borrowed-demand signal.

In [3]:
import sys

sys.path.insert(0, "../src")
from signals import add_derived_signals

# Stage 2 reads observable fields only.
signals_df = add_derived_signals(df[observable_cols])

derived_cols = [
    "engagement_intensity",
    "purchase_intent",
    "discount_dependency",
    "early_timing",
]

# Show the observable inputs alongside the derived signals only - latent and
# long-term columns are never loaded here.
signals_df[["user_id", "group", "made_first_purchase_30d"] + derived_cols].head()

,user_id,group,made_first_purchase_30d,engagement_intensity,purchase_intent,discount_dependency,early_timing
0,1,treatment,0,0.185433,0.67820,0.329300,NaN
1,2,control,0,0.121683,0.49035,0.059200,NaN
2,3,treatment,1,0.673283,0.18905,0.873000,0.172648
3,4,treatment,0,0.743200,0.89095,0.174725,NaN
4,5,control,1,0.864067,0.67820,0.059200,0.786372


Each derived signal is a percentile within the cohort, so it reads as *where a
user sits relative to everyone else* rather than a raw count. High
`engagement_intensity` and `purchase_intent` point toward genuine interest;
high `discount_dependency` points toward discount-led behaviour; `early_timing`
simply flags an early purchase, which only becomes meaningful **in combination**
with the other signals in Stage 3.

## Stage 3: Early Read Model

Stage 3 turns the early signals into an initial read on customer quality and
borrowed-demand risk. This component is **intentionally modular**: the v1 below
is a transparent, rule-based read, and it could later be swapped for a
statistical or ML model without changing the rest of the framework, as long as
it emits the same read columns.

It follows a "score, then band" shape. Continuous scores are computed
internally as an intermediate step, then exposed as a coarse **high / medium /
low** read using within-cohort tertiles — no hand-picked thresholds, and no
false precision.

- **quality_score** — high engagement + high intent + low discount dependency.
- **borrowed_risk_score** — early purchase **and** weak engagement **and** high
  discount dependency, combined as a product so *all three* must be present.
  Buying early on its own does not raise it. Purchasers only.
- **discount_dependency_score** — the `discount_dependency` signal directly.

In [4]:
from early_read import add_early_read

read_df = add_early_read(signals_df)

score_cols = ["quality_score", "borrowed_risk_score", "discount_dependency_score"]
read_cols = ["quality_read", "borrowed_risk_read", "discount_dependency_read"]

# Internal scores are intermediate; the high/medium/low reads are what we act on.
read_df[["user_id", "group"] + score_cols + read_cols].head()

,user_id,group,quality_score,borrowed_risk_score,discount_dependency_score,quality_read,borrowed_risk_read,discount_dependency_read
0,1,treatment,0.511444,NaN,0.329300,medium,<NA>,medium
1,2,control,0.517611,NaN,0.059200,medium,<NA>,low
2,3,treatment,0.329778,0.049243,0.873000,low,medium,high
3,4,treatment,0.819808,NaN,0.174725,high,<NA>,medium
4,5,control,0.827689,0.006328,0.059200,high,low,low


### Cohort-level Read

The read is summarised at the **cohort level**, not per user. We compare the
*mix* of high / medium / low reads among **purchasers** in each arm. This keeps
the output a decision-useful read on the cohort rather than an individual-level
label or prediction.

In [5]:
purchasers = read_df[read_df["made_first_purchase_30d"] == 1]
levels = ["high", "medium", "low"]

parts = []
for col in read_cols:
    mix = (
        purchasers.groupby("group")[col]
        .value_counts(normalize=True)
        .unstack("group")
        .reindex(index=levels)
    )
    mix.index = pd.MultiIndex.from_product([[col], mix.index], names=["read", "level"])
    parts.append(mix)

# Share (%) of each arm's purchasers falling in each read band, plus the gap.
cohort_summary = pd.concat(parts) * 100
cohort_summary["delta_pp"] = cohort_summary["treatment"] - cohort_summary["control"]
cohort_summary.round(1)

group                            control  treatment  delta_pp
read                     level                               
quality_read             high       70.2       36.2     -34.0
                         medium     22.7       27.3       4.5
                         low         7.1       36.5      29.4
borrowed_risk_read       high       10.7       50.5      39.8
                         medium     40.4       27.9     -12.5
                         low        48.9       21.6     -27.3
discount_dependency_read high       21.4       72.7      51.3
                         medium     58.2       20.4     -37.8
                         low        20.4        6.8     -13.6

**Reading the cohort.** Compared with control purchasers, treatment purchasers
show a much higher share of **high** discount-dependency reads (about 73% vs
21%) and **high** borrowed-risk reads (about 51% vs 11%), alongside a much
lower share of **high**-quality reads (about 36% vs 70%). Taken together, that
is a decision-useful read that a meaningful part of the headline lift may be
carried by discount-led, pulled-forward purchases rather than durable
incremental demand.

This stays at the cohort level on purpose. It is a read on the *mix* of the
treatment purchaser cohort — not a label on any individual user, and not a
prediction of any one user's long-term value. It points to where the risk
likely sits. Stage 4 turns this into a scale / adjust / stop recommendation,
and Stage 5 checks the read against mature long-term outcomes.

## Stage 4: Decision Tree

Stage 4 upgrades the earlier "decision mapping" idea into an explicit **decision
tree**: the experiment passes through ordered **gates**, and each gate either
stops the flow or hands control to the next. This makes the reasoning auditable —
you can see exactly where a decision was made and why.

- **Gate 0 — Experiment Readiness and Data Quality Check.** Computable checks on
  sample ratio, duplicate users, missingness, coupon logic, timing validity, and
  metric availability. Any failure stops the flow: *do not decide on bad data.*
- **Gate 1 — Primary Metric Check.** Did the 30-day first-purchase rate move? This
  only asks whether the short-term target moved, not whether the lift is durable
  or valuable.
- **Gate 2 — Guardrail and Dependency Check.** Discount dependency as an *early
  risk proxy* — not a final cost metric (true cost is settled in Stage 5).
- **Gate 3 — Early Quality Read.** The treatment purchaser quality *mix*.
- **Gate 4 — Borrowed-Demand Risk Read.** The treatment purchaser borrowed-risk
  *mix* (a combined signal, not "bought early" alone).
- **Gate 5 — Decision Output.** Combine the reads into: Scale · Scale selectively
  / Adjust · Adjust · Continue measurement · Stop.

Stage 4 still uses only observable fields and the Stage 3 reads — no latent or
long-term outcome fields.

### Decision Rule Summary

| Situation | Framework decision | Operating logic |
|---|---|---|
| Data quality / readiness check fails | Do not decide yet | Fix instrumentation, assignment, or data quality before reading the experiment |
| No primary lift | Stop or redesign offer | The short-term target did not move |
| Lift + strong quality + low borrowed risk + acceptable dependency | Scale | Broaden rollout while keeping holdout measurement |
| Lift + weak quality + high borrowed risk + high/elevated dependency | Adjust, not broad scale | Redesign targeting or incentive before any broad rollout |
| Lift + inconclusive signals | Continue measurement | Extend observation window or add missing signals |
| Lift + mixed evidence | Scale selectively / Adjust | Use judgment, target stronger segments, keep holdout, and monitor |

Extreme cases have clear rules. Mixed cases are intentionally routed to selective rollout, adjustment, or further measurement rather than being over-automated. The framework provides guardrails; it does not assume every business decision can be fully automated by a lookup table.

In [6]:
from decision_tree import run_decision_tree, format_report

# Stage 4 runs on the Stage 3 read dataframe: observable fields + early reads only.
result = run_decision_tree(read_df)
print(format_report(result))

STAGE 4 - DECISION TREE
[PASS          ] Gate 0: Experiment Readiness and Data Quality Check
    Data quality acceptable; all readiness checks pass.
      - [ok  ] sample_ratio: control share 0.501 (control=5,015, treatment=4,985); within +/-2% tol=True; chi2=0.09 (<3.841 -> balanced=True)
      - [ok  ] duplicate_user: 0 duplicate user_id rows
      - [ok  ] missingness: max missing rate 0.000% in 'user_id' (threshold 5%)
      - [ok  ] coupon_logic: 0 control-group rows have coupon_used=1 (expected 0)
      - [ok  ] timing_validity: non-purchasers with a purchase day: 0; purchasers missing a day: 0
      - [ok  ] metric_availability: all primary, Stage 2 signal, and Stage 3 read columns present
[PASS (lift)   ] Gate 1: Primary Metric Check
    30d first-purchase rate: control 17.71% vs treatment 23.55% (lift +5.84pp). Primary lift exists.
[CONTINUE      ] Gate 2: Guardrail and Dependency Check
    Discount-dependency risk proxy: 72.7% of treatment purchasers read 'high' vs 21.4% of c

**Reading this case.** Gate 0 passes (data quality acceptable). Gate 1: the
primary lift exists (**+5.84pp**). Gate 2: discount dependency is **high** (about
73% of treatment purchasers read high dependency vs 21% of control). Gate 3: the
treatment purchaser quality mix is **weaker** (about 36% high-quality vs 70% in
control). Gate 4: the borrowed-risk read is **higher** (about 51% vs 11%).

**Decision: Adjust, not broad scale.** Do not scale broadly yet. Adjust targeting
or incentive design, keep a holdout, and continue tracking long-term value.

The lift is real, so this is not a *Stop*. But the early reads say a meaningful
share of the lift is carried by discount-led, pulled-forward purchases rather than
durable incremental demand — an unfavorable quality/risk mix — so it is not a
broad *Scale* either. The defensible move is to refine the offer and targeting,
hold out a control, and let long-term value mature before any broad rollout.

## Framework Flow

The structure stays fixed across experiments; the method inside each stage can
improve over time. Stage 4 is where the flow branches into a decision, and
Stage 5 feeds what it learns back into the next cycle.

```
Stage 1  Measurement Design
   |  primary metric, guardrails, holdout, observation windows, post-period tracking
   v
Stage 2  Early Signal Layer
   |  observable behaviour -> transparent derived signals (0-1 percentile scale)
   v
Stage 3  Early Read Model
   |  signals -> high / medium / low reads (quality, borrowed risk, discount dependency)
   v
Stage 4  Decision Tree
   |  Gate 0  data quality ----fail----> do not decide; fix instrumentation
   |  Gate 1  primary lift ----none----> Stop / redesign offer
   |  Gate 2  discount dependency (early risk proxy)
   |  Gate 3  quality mix
   |  Gate 4  borrowed-risk mix
   |  Gate 5  combine --> Scale | Scale selectively / Adjust | Adjust | Continue | Stop
   v
Stage 5  Backtest, Calibration & Monitoring
   |  5A backtest reads vs mature long-term outcomes
   |  5B next-cycle operating update (stage-by-stage changes for the next cycle)
   |  5C post-launch monitoring design (mode the framework would enter if scaled)
   v
Next experiment  (improved signals, gates, and targeting)
```

## Stage 5: Backtest, Calibration, and Monitoring Loop

Stage 5 closes the loop once long-term outcomes mature. It has three parts:

- **5A — Backtest.** Compare the Stage 3 early reads against mature long-term
  outcomes: did the cohorts we read early as high quality / high risk / high
  dependency actually behave that way?
- **5B — Next-cycle Operating Update.** Turn the backtest readout into concrete,
  stage-by-stage updates for the next experiment cycle.
- **5C — Post-launch Monitoring Design.** Define the monitoring mode the framework
  *would enter* if a strategy were scaled — tracked on rolling cohort data over
  time, not on this single snapshot.

5A and 5B run on this dataset. 5C is a monitoring **design**: the current data is
a single experiment snapshot, so that section specifies the monitoring mode the
framework would enter rather than showing monitoring results.

### Stage 5A: Backtest — Do Early Reads Align with Later Outcomes?

This is the **first** point in the walkthrough where we open the long-term
outcome fields (`repeat_purchase_180d`, `orders_180d`, `net_revenue_180d`,
`realized_ltv_180d`, …). Stages 1–4 never loaded them.

The backtest groups purchasers by their early read **band** and looks at the
average long-term outcome in each band. The question is whether the early reads
**align** with what actually happened — a directional check on the read
*instrument*, not a claim that the reads *predict* any individual user's outcome.

For the borrowed-risk table we first derive **post-period orders**:

```
post_period_orders_31_180d = orders_180d − made_first_purchase_30d
```

`orders_180d` includes the first purchase, so we subtract it to isolate demand
that landed *after* the 30-day first-purchase window — the part borrowed demand
is really about.

*Cohort: all purchasers, both arms pooled.*

In [7]:
from backtest import run_backtest, LONG_TERM_FIELDS

# Stage 5 opens the long-term fields and joins them onto the Stage 3 reads.
# (run_backtest derives post_period_orders_31_180d internally.)
backtest_df = read_df.merge(df[["user_id", *LONG_TERM_FIELDS]], on="user_id")
tables = run_backtest(backtest_df)

print("Table 1 — Quality read vs long-term value")
display(tables["quality_vs_value"])
print("Table 2 — Borrowed-risk read vs long-term outcome")
display(tables["borrowed_risk_vs_outcome"])
print("Table 3 — Discount-dependency read vs long-term value")
display(tables["discount_dependency_vs_value"])

Table 1 — Quality read vs long-term value


,repeat_purchase_180d,orders_180d,realized_ltv_180d
quality_read,,,
high,0.91,3.92,205.85
medium,0.58,2.32,86.44
low,0.30,1.42,26.94


Table 2 — Borrowed-risk read vs long-term outcome


,post_period_orders_31_180d,net_revenue_180d,realized_ltv_180d
borrowed_risk_read,,,
high,0.69,46.34,46.34
medium,2.06,140.58,140.58
low,3.01,211.99,211.99


Table 3 — Discount-dependency read vs long-term value


,repeat_purchase_180d,net_revenue_180d,realized_ltv_180d
discount_dependency_read,,,
high,0.52,86.00,86.00
medium,0.85,173.34,173.34
low,0.83,203.43,203.43


**Reading the three tables (all directional, not predictive).**

- **Table 1 — quality read vs value.** The high-quality-read band shows the
  highest repeat rate, order count, and realized LTV, stepping down through medium
  to low. The early quality read *aligns* with later value.
- **Table 2 — borrowed-risk read vs outcome.** The high-risk band has the
  **weakest** post-period (day 31–180) orders and the lowest net revenue / LTV,
  while the low-risk band has the strongest. The borrowed-risk read *aligns* with
  weaker post-period demand.
- **Table 3 — discount-dependency read vs value.** The high-dependency band shows
  the **lowest** repeat rate and net revenue / LTV. High discount dependency
  *aligns* with weaker long-term value.

Together these *validate the direction* of the early reads on this dataset: the
cohorts flagged early went on to behave as flagged. This is consistency with
later outcomes, not a predictive model — but it is enough to justify keeping
these reads, and (per Stage 5B) strengthening the discount-dependency guardrail,
in the next cycle.

### Stage 5B: Next-cycle Operating Update

Calibration is where the loop closes: each thing this cycle showed becomes a
concrete change to a specific stage of the *next* cycle. The table below is an
**operating update**, not automatic weight tuning or threshold fitting. It reads
across five columns, from what we observed this cycle through to exactly what each
stage does differently next time.

| This cycle observation | Framework interpretation | Next Stage 1 update | Next Stage 2/3 update | Next Stage 4 update |
|---|---|---|---|---|
| Treatment lift exists, but treatment purchasers show weaker quality mix | Coupon has short-term lift, but broad scaling is not justified yet | Predefine target segments before next test | Compare early quality mix by segment | Lift alone cannot trigger Scale |
| High discount dependency aligns with weaker long-term value in this simulated case | Discount dependency is an important early risk proxy | Add discount dependency as a stronger guardrail | Keep discount dependency read; consider adding full-price repeat signal | Gate 2 becomes a stronger guardrail before Scale |
| High borrowed-risk read aligns with weaker post-period orders in this walkthrough | Borrowed-risk read should limit broad rollout until longer outcomes mature | Keep holdout and define 31-180d tracking window | Keep timing × weak engagement × discount dependency as combined risk logic | High borrowed risk blocks broad Scale |
| Early timing alone is insufficient | Fast conversion is not automatically bad | Track timing, but do not use it as a standalone decision metric | Timing remains only one input in the combined risk score | Gate 4 cannot be triggered by timing alone |
| Current data lacks post-purchase engagement | The next test needs stronger early quality evidence | Add post-purchase engagement tracking after first order | Add post-purchase sessions / repeat browsing as new signals | Scale condition becomes stricter if post-purchase engagement is weak |

This table is the operating process, not precise parameter tuning. It shows how
this cycle's result turns into specific, auditable adjustments to each stage of
the next cycle — predefining segments and post-purchase tracking in Stage 1,
adding signals in Stage 2/3, and tightening the Stage 4 gates so that lift or
timing alone can never trigger a broad Scale. That is the loop closing: what this
cycle learned becomes next cycle's improvement, stage by stage.

### Stage 5C: Post-launch Monitoring Design

The current dataset is a single experiment snapshot. It supports experiment
readout and backtest, but not empirical post-launch decay monitoring. This
section defines the monitoring mode the framework would enter if a strategy were
scaled.

Everything below is a **design / template**: it specifies *what* would be tracked
and *how* the monitoring decision tree would run on rolling cohort data over time.
It is not a monitoring result, and it does not compute decay from this single
snapshot.

The monitoring design tracks three families of read, all on rolling cohort data
over time:

- **A. Effect Decay** — rolling lift and the treatment-vs-holdout gap: *is the
  coupon still incremental?*
- **B. Quality Decay** — repeat rate, realized LTV, and high-quality read share:
  *is acquired quality holding?*
- **C. Dependency / Fatigue** — discount dependency, coupon-used share, and
  full-price repeat: *are users becoming discount-trained?*

These reuse the Stage 3 reads (quality / borrowed risk / discount dependency);
monitoring simply tracks them across periods instead of reading them once.

In [8]:
import sys

sys.path.insert(0, "../src")
from monitoring import validate_monitoring_schema, evaluate_monitoring_state

# Run the monitoring schema check on the CURRENT snapshot (the same df loaded in
# Stage 1). No rolling data is fabricated: the snapshot is a single cross-section,
# so the expected outcome is "not monitoring-ready".
check = validate_monitoring_schema(df)
print(check.message)
print("missing rolling keys:", check.missing_rolling_keys)

# The combined evaluation reports the same boundary, not a decay read.
print("monitoring state    :", evaluate_monitoring_state(df).state)

not monitoring-ready: missing rolling fields (period, strategy_version). This dataset is a single experiment snapshot and is not monitoring-ready.
missing rolling keys: ['period', 'strategy_version']
monitoring state    : not monitoring-ready


The check reports **not monitoring-ready: missing rolling fields (period,
strategy_version)**. This is the framework stating its own data boundary: a
single experiment snapshot is enough for the Stage 5A backtest workflow, but
monitoring needs rolling cohort data over time — the same strategy measured
across successive periods and versions. Rather than forcing a decay number out of
cross-sectional data, the framework reports that the data is not monitoring-ready.
That honest boundary is the point of this section: the framework knows that a
single experiment snapshot is not enough to monitor decay, and says so.

**Monitoring Decision Tree.** If a strategy were scaled and rolling cohort data
were flowing, monitoring would run this tree each period:

```
Strategy is running (scaled)
  |
  v
1. Incremental lift still positive vs holdout?
   No  -> Pause or reduce rollout; re-estimate targeting/incentive
   Yes |
       v
2. User quality stable?
   No  -> Narrow targeting; reduce exposure to low-quality segments
   Yes |
       v
3. Discount dependency increasing?
   Yes -> Adjust incentive (lower discount / higher threshold / personalized offer)
   No  |
       v
4. Borrowed-demand risk increasing?
   Yes -> Extend measurement window; maintain holdout; avoid broad scale
   No  |
       v
Continue running, with rolling monitoring
```

**Same framework, two modes.** This monitoring tree uses the same reads as Stage
4 — quality, borrowed risk, and discount dependency — only now they are tracked
as trends across periods instead of being read once. The framework therefore
covers both modes with one structure: a one-time experiment readout in Stages
1–4, and, once a strategy is scaled, the post-launch monitoring mode designed
here. The decision logic does not change; only the time axis is added.